In [ ]:
# import the required libraries
import pandas as pd
import torch
import json

In [ ]:
# read the input data (cleaned embedded keywords-title-overview dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title-overview dataset is the dataset 
formed by merging & cleaning the embedded keywords, embedded title and embedded overview dataset.
"""
movies_keywords_genres = pd.read_excel("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/pre-final_mkg_dataset_gpt2.xlsx")

In [ ]:
movies_keywords_genres.head(1)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,2003-06-29 00:00:00,Mongolian,7.07,100,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...,[-7.90620819e-02 1.27022997e-01 -7.71413565e-...


In [ ]:
movies_keywords_genres.shape[0]

234762

In [ ]:
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,2003-06-29 00:00:00,Mongolian,7.07,100,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...,[-7.90620819e-02 1.27022997e-01 -7.71413565e-...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,2005-09-15 00:00:00,"German, French",6.3,26,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...,[-1.14440762e-01 -7.02230632e-02 -2.19043717e-...
2,False,1116.0,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,2006-06-23 00:00:00,"Irish, Latin, English",7.4,747,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...,[-1.15996398e-01 -2.08327860e-01 -7.92207662e-...
3,False,1123.0,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,2006-10-27 00:00:00,"Afrikaans, English, Zulu",6.215,114,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...,[-2.46082172e-01 1.04989491e-01 -3.55010718e-...
4,False,1124.0,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,2006-10-19 00:00:00,English,8.203,15214,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...,[ 6.16751704e-03 -1.18230581e-01 -7.90594935e-...


In [ ]:
movies_keywords_genres['overview_embedding_string'] = movies_keywords_genres['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

In [ ]:
movies_keywords_genres.loc[0, 'overview_embedding_string']

'-7.90620819e-02  1.27022997e-01 -7.71413565e-01 -9.57237408e-02 -1.98667616e-01 -1.43106177e-01  1.47545946e+00 -1.46336094e-01  2.09725097e-01  2.01271579e-01  6.13774598e-01 -2.42962018e-01 -1.20739460e-01 -9.45630744e-02 -2.46817078e-02  2.44118378e-01  2.24141732e-01  2.90502310e-02  2.40911886e-01 -1.83276698e-01  3.87563139e-01 -1.34887502e-01 -2.12923929e-01 -1.08384244e-01  2.81496137e-01  2.80038744e-01  1.71093139e-04  2.14782774e-01 -3.69584337e-02  5.46785817e-02 -1.51102897e-02 -1.52639505e-02 -1.12024054e-01  4.47068959e-02 -1.11463457e-01  3.52249593e-01  7.42878265e+01  9.69035327e-02 -1.12410896e-01 -8.03367421e-02 -2.11347770e-02  1.09749874e-02 -2.69903690e-01 -8.47044773e-03 -4.59584206e-01 -2.71406293e-01  1.03832766e-01  4.58366014e-02  3.67437042e-02  1.76635519e-01  1.79629937e-01  2.99487948e-01  2.18376800e-01  1.14052691e-01  1.47575065e-01  1.31625795e+00  2.09961370e-01  4.29361686e-02 -2.33544439e-01  2.52068430e-01 -1.12367988e-01 -4.61391695e-02  2.1014

In [ ]:
movies_keywords_genres.head(2)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding,overview_embedding_string
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,2003-06-29 00:00:00,Mongolian,7.07,100,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...,[-7.90620819e-02 1.27022997e-01 -7.71413565e-...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,2005-09-15 00:00:00,"German, French",6.3,26,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...,[-1.14440762e-01 -7.02230632e-02 -2.19043717e-...,-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...


In [ ]:
drop_columns = ['overview_embedding']
movies_keywords_genres = movies_keywords_genres.drop(columns=drop_columns)
movies_keywords_genres.head(5)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string,overview_embedding_string
0,False,1114.0,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,2003-06-29 00:00:00,Mongolian,7.07,100,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...
1,False,1115.0,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,2005-09-15 00:00:00,"German, French",6.3,26,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...,-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...
2,False,1116.0,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,2006-06-23 00:00:00,"Irish, Latin, English",7.4,747,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...
3,False,1123.0,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,2006-10-27 00:00:00,"Afrikaans, English, Zulu",6.215,114,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...
4,False,1124.0,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,2006-10-19 00:00:00,English,8.203,15214,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...,6.16751704e-03 -1.18230581e-01 -7.90594935e-0...


In [ ]:
nan_count = movies_keywords_genres['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding_string' has {nan_count} NaN values.")
nan_count = movies_keywords_genres['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
movies_keywords_genres.dropna(subset=['overview_embedding_string'], inplace=True)
movies_keywords_genres.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding_string' has 50 NaN values.
The column 'genres' has 37 NaN values.


In [ ]:
movies_keywords_genres.shape[0]

234712

In [ ]:
movies_keywords_genres.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/final_mkg_dataset_gpt2.csv', index=False)

In [ ]:
# this is the final embedded dataset
tedf = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/final_mkg_dataset_gpt2.csv")

In [ ]:
tedf.shape[0]

234712